Objective is to show that embeddings captured by ATOMICA cannot be completely recapitulated by other embedding models

In [24]:
import pickle
import numpy as np 
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
import torch

In [7]:
molecule_embddings_path = "/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/latent_space/Transformer-M/pl_test_unsanitized_embeddings_transformer_m_18.pt"
prot_embeddings_path = "/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/latent_space/ESM/PL_test_ESM2_embeddings.pkl"
atomica_embeddings_path = "/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/latent_space/embeddings/PL_test_epoch147_step1581725.pkl"

In [26]:
molecule_embeddings = torch.load(molecule_embddings_path)
with open(atomica_embeddings_path, "rb") as f:
    atomica_embeddings = pickle.load(f)
with open(prot_embeddings_path, "rb") as f:
    prot_embeddings = pickle.load(f)

In [27]:
embeddings_ids = [x['id'] for x in atomica_embeddings]
atomica_embeddings_matrix = np.array([x['graph_embedding'] for x in atomica_embeddings])
molecule_embeddings_matrix = molecule_embeddings['embeds'][[molecule_embeddings['ids'].index(x) for x in embeddings_ids]].numpy()
prot_embeddings_matrix = np.array([prot_embeddings[x] for x in embeddings_ids])

In [ ]:
atomica_embeddings_sim_matrix = cosine_similarity(atomica_embeddings_matrix)
molecule_embeddings_sim_matrix = cosine_similarity(molecule_embeddings_matrix)

In [25]:
spearmanr(atomica_embeddings_sim_matrix.flatten(), molecule_embeddings_sim_matrix.flatten())

SignificanceResult(statistic=0.10722915454872758, pvalue=0.0)

In [28]:
prot_embeddings_sim_matrix = cosine_similarity(prot_embeddings_matrix)
spearmanr(atomica_embeddings_sim_matrix.flatten(), prot_embeddings_sim_matrix.flatten())

SignificanceResult(statistic=-0.1439637417438651, pvalue=0.0)

In [29]:
prot_molecule_embeddings_matrix = np.concatenate([prot_embeddings_matrix, molecule_embeddings_matrix], axis=1)
prot_molecule_embeddings_sim_matrix = cosine_similarity(prot_molecule_embeddings_matrix)
spearmanr(atomica_embeddings_sim_matrix.flatten(), prot_molecule_embeddings_sim_matrix.flatten())

SignificanceResult(statistic=-0.143457681452268, pvalue=0.0)

In [33]:
combined_cosine_similarity = (prot_embeddings_sim_matrix + molecule_embeddings_sim_matrix) / 2
spearmanr(atomica_embeddings_sim_matrix.flatten(), combined_cosine_similarity.flatten())

SignificanceResult(statistic=0.08742164759151407, pvalue=0.0)

In [32]:
prot_embeddings_matrix.shape, molecule_embeddings_matrix.shape

((6654, 2560), (6654, 768))